In [1]:
import pandas as pd

total_pit = pd.read_csv("../../data/processed/total_pit.csv")

In [2]:
total_pit.head(5)

,Unnamed: 0,Unnamed: 0.1,iso3,prod_level,alloc_key,cell5m,x,y,rec_type,tech_type,...,rcof_s,coco_s,teas_s,toba_s,bana_s,plnt_s,trof_s,temf_s,vege_s,rest_s
0,0,0,ind,CH08078,4383640,1891479,123.291667,53.541667,P,I,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,CHN,CH08078,4393627,1895786,122.208333,53.458333,P,I,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2,CHN,CH08078,4393628,1895787,122.291667,53.458333,P,I,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,3,CHN,CH08078,4393629,1895788,122.375000,53.458333,P,I,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,4,CHN,CH08078,4393637,1895796,123.041667,53.458333,P,I,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
832827 / (4320 * 2160)

0.08925186471193415

# Zonal crop data processing

This notebook contains the methodology for processing crop data contained in [paper]. The objective of this analysis to map crop production across the most common 172 crops onto the world's countries.

In [2]:
import os
from allfed_spatial.features.io import load_features, write_features

### Resources
The raw datasets used in this analysis are as follows:

In [27]:
EARTHSTAT_CROP_DATA = ""
# COUNTRY_BOUNDARY_DATA = "https://osf.io/p93yg/download"
COUNTRY_BOUNDARY_DATA = "/Users/tim/allfed/residues_ruminants_project/gaul_all_countries_dissolved_4326.gpkg"

### Spatially join crop production with country boundaries using `zonal_stats`

`zonal_stats` from the `rasterstats` library gives us an easy way to summarise raster statistics across an iterable of vector geometries. We'll use this to create attributes on our country boundaries which describe the production of each of the 172 crops we have data for.

In [50]:
# Download and unzip crop data (note: requires ~3GB of storage)
#TODO download/unzip
base_crop_dir = '/Users/tim/allfed/data/landuse_yield/earthstat/HarvestedAreaYield175Crops_Geotiff/GeoTiff'
crop_dirs = [dirs for subdir, dirs, files in os.walk(base_crop_dir)][0]

In [55]:
# Download country boundaries and map to geometries
countries = load_features(COUNTRY_BOUNDARY_DATA)
country_geometries = [c.geom for c in countries]

In [56]:
# Iterate through crop folder, and run the production raster in each through
# zonal_stats on our country boundaries, assigning the result to the associated
# countries feature data.
for i, crop_type in enumerate(crop_dirs):
    print(f'Processing {i+1}/{len(crop_dirs)}: {crop_type}')
    stats = zonal_stats(
        country_geometries,
        '{}/{}/{}_Production.tif'.format(base_crop_dir, crop_type, crop_type),
        stats=['sum']
    )
    for j, s in enumerate(stats):
        countries[j].update_data(
            f'{crop_type}_sum', 
            round(s['sum'] if s['sum'] else 0.0, 3)
        )

Processing 1/172: apple


In [45]:
# Write data
write_features(countries, "../data/processed/global_crop_boundaries.gpkg")

### Create crop data based on top crops

In [1]:
# We want a spatial depiction of the global availability of residues - do this as a raster
# 1. Load crops we have data for in
# 2. Run residue calc using lookup
# 3. Produce rasters of residues, energy, protein, 


# ME is in MJ/kg DM
# Protein is in percent of DM
# same with NDF

In [2]:
import pandas as pd
import os
import rasterio
import numpy as np
from functools import partial

base_crop_dir = '/Users/tim/allfed/data/landuse_yield/earthstat/HarvestedAreaYield175Crops_Geotiff/GeoTiff'
lookup = pd.read_csv("../data/processed/residue_lookup.csv", index_col="crop")
crop_dirs = [dirs for subdir, dirs, files in os.walk(base_crop_dir)][0]

def get_dry_crop_residue_mass(crop_production, crop_name, lookup):
    if crop_name in lookup.index:
        res_tonnes = lookup.loc[crop_name].sg_ratio * crop_production * lookup.loc[crop_name].res_dm_ratio
        return res_tonnes
    return 0

def get_rum_me_from_residue_mass(res, crop_name, lookup):
    if crop_name in lookup.index:
        # convert tonnes to kg, multiply by ruminant metabolisable energy
        # this is in MJ
        energy_mj = res * 1000 * lookup.loc[crop_name].rum_me
        return energy_mj
    return 0
    
def get_crude_protein_from_residue_mass(res, crop_name, lookup):
    if crop_name in lookup.index:
        # convert from percent to fraction, get tonnes of protein
        protein_tonnes = res * (lookup.loc[crop_name].crude_protein_perc / 100)
        return protein_tonnes
    return 0
    
def get_ndf_from_residue_mass(res, crop_name, lookup):
    if crop_name in lookup.index:
        # convert from percent to fraction, get tonnes of fibre
        fibre_tonnes = res * (lookup.loc[crop_name].nd_fibre / 100)
        return fibre_tonnes
    return 0

def convert_mj_to_mj_per_ha(mj):
    """ Assume pixel size is 10x10 km """
    return mj / 10000

def convert_tonnes_to_tonnes_per_ha(tonnes):
    """ Assume pixel size is 10x10 km """
    return tonnes / 10000


lookup.head(5)

,sg_ratio,res_dm_ratio,crude_protein_perc,rum_me,nd_fibre,code
crop,,,,,,
wheat,1.333333,0.910,4.2,6.8,77.5,1
rice,1.200000,0.920,8.2,7.9,70.0,2
maize,0.892857,0.289,6.9,9.3,65.0,3
alfalfa,1.000000,0.906,18.3,8.5,45.9,4
sugarcane,0.250000,0.260,5.5,8.0,70.0,5


In [3]:
residue_arrays = []
me_arrays = []
protein_arrays = []
ndf_arrays = []

residue_data = lookup.copy()
residue_data["total_production"] = np.nan
residue_data["total_residues"] = np.nan
residue_data["total_crude_protein"] = np.nan
residue_data["total_rum_me"] = np.nan
residue_data["total_ndf"] = np.nan

width = 4320
height = 2160

for i, crop_type in enumerate(crop_dirs):
    
    if crop_type in lookup.index:
        
        # Read crop production raster
        prod_file = '{}/{}/{}_Production.tif'.format(base_crop_dir, crop_type, crop_type)
        dataset = rasterio.open(prod_file)
        prod_arr = dataset.read(1)
#         combined_production_dataset.write(prod_arr, int(lookup.loc[crop_type].code))
        residue_data.at[crop_type, "total_production"] = np.sum(prod_arr)
        
        # Calculate residues
        res_func = partial(get_dry_crop_residue_mass, crop_name=crop_type, lookup=lookup)
        # get residue tonnes
        res_arr = res_func(prod_arr)
        residue_data.at[crop_type, "total_residues"] = np.sum(res_arr)
        # get residue tonnes / ha
        res_arr_per_ha = convert_tonnes_to_tonnes_per_ha(res_arr)
        residue_arrays.append(res_arr_per_ha)
        
        # Calculate ME from residues
        me_func = partial(get_rum_me_from_residue_mass, crop_name=crop_type, lookup=lookup)
        # get ME MJ
        me_arr = me_func(res_arr)
        residue_data.at[crop_type, "total_rum_me"] = np.sum(me_arr)
        # get ME MJ / ha
        me_arr_per_ha = convert_mj_to_mj_per_ha(me_arr)
        me_arrays.append(me_arr_per_ha)
        
        # Calculate protein from residues
        protein_func = partial(get_crude_protein_from_residue_mass, crop_name=crop_type, lookup=lookup)
        # get protein tonnes
        protein_arr = protein_func(res_arr)
        residue_data.at[crop_type, "total_crude_protein"] = np.sum(protein_arr)
        # get protein tonnes / ha
        protein_arr_per_ha = convert_tonnes_to_tonnes_per_ha(protein_arr)
        protein_arrays.append(protein_arr_per_ha)
        
        
        # Calculate NDF from residues
        ndf_func = partial(get_ndf_from_residue_mass, crop_name=crop_type, lookup=lookup)
        # get ndf tonnes
        ndf_arr = ndf_func(res_arr)
        residue_data.at[crop_type, "total_ndf"] = np.sum(ndf_arr)
        # get ndf tonnes / ha
        ndf_arr_per_ha = convert_tonnes_to_tonnes_per_ha(ndf_arr)
        ndf_arrays.append(ndf_arr_per_ha)
        
        # Initiate output datasets
        if crop_type == "wheat":
            transform = dataset.transform
            dtype = prod_arr.dtype
            summed_residues_dataset = rasterio.open(
                "../data/processed/residue_inventory/residues_dm.tif",
                "w",
                driver="GTiff",
                height=height,
                width=width,
                count=1,
                dtype=dtype,
                crs='+proj=latlong',
                transform=transform
            )
            summed_me_dataset = rasterio.open(
                "../data/processed/residue_inventory/ruminant_me.tif",
                "w",
                driver="GTiff",
                height=height,
                width=width,
                count=1,
                dtype=dtype,
                crs='+proj=latlong',
                transform=transform
            )
            summed_protein_dataset = rasterio.open(
                "../data/processed/residue_inventory/crude_protein.tif",
                "w",
                driver="GTiff",
                height=height,
                width=width,
                count=1,
                dtype=dtype,
                crs='+proj=latlong',
                transform=transform
            )
            summed_ndf_dataset = rasterio.open(
                "../data/processed/residue_inventory/ndf.tif",
                "w",
                driver="GTiff",
                height=height,
                width=width,
                count=1,
                dtype=dtype,
                crs='+proj=latlong',
                transform=transform
            )
    #         combined_production_dataset = rasterio.open(
    #             "../data/processed/residue_inventory/crop_production.tif",
    #             "w",
    #             driver="GTiff",
    #             height=height,
    #             width=width,
    #             count=29,
    #             dtype=new_arr.dtype,
    #             crs='+proj=latlong',
    #             transform=dataset.transform
    #         )
        
        
summed_residues = sum(residue_arrays)
summed_residues_dataset.write(summed_residues, 1)
summed_residues_dataset.close()

summed_me = sum(me_arrays)
summed_me_dataset.write(summed_me, 1)
summed_me_dataset.close()

summed_protein = sum(protein_arrays)
summed_protein_dataset.write(summed_protein, 1)
summed_protein_dataset.close()

summed_ndf = sum(ndf_arrays)
summed_ndf_dataset.write(summed_ndf, 1)
summed_ndf_dataset.close()

# combined_production_dataset.close()

In [5]:
residue_data.to_csv("../data/processed/residue_output_table.csv")

In [12]:
# plotting

import plotly
import cufflinks as cf
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot


In [17]:
# make figure
fig = residue_data[["total_production", "total_residues"]].iplot(asFigure=True, kind='bar', barmode = 'stack',
               xTitle='Crops',yTitle='numbers',title='Returns')

# plot figure
iplot(fig)